# 04 — Prithvi-EO-2.0-300M Head-Capacity Ablation
**Runtime:** Google Colab — **GPU (T4 minimum, A100 preferred)**

**Purpose:** Head-capacity ablation on frozen Prithvi-300M embeddings (1024-d).  
Entire encoder is frozen; only the lightweight regression head is trained.

Two heads are compared:

| Head | Architecture | Trainable params |
|---|---|---|
| **Linear** | LayerNorm(1024) → Dropout(0.1) → Linear(1024, 1) | ~1 K |
| **MLP** | LayerNorm → Linear(1024,256) → GELU → Dropout → Linear(256,128) → GELU → Dropout → Linear(128,1) | ~296 K |

Two targets per head:

| Target | Formula | Interpretation |
|---|---|---|
| `coverage_fraction` | n_ookla_tiles / total_possible | Spatial coverage |
| `log_mean_tests` | log(1 + total_tests / n_subtiles) | Test density per sub-tile |

Predictions are binarised with a val-optimal threshold and evaluated against binary
ground-truth labels on the Northeast India geographic hold-out.

**Inputs:**
- `data/raw/patches_2019/` — 6,970 GeoTIFF patches (224×224×6 HLS bands)
- `data/processed/patches_with_splits.csv` — patch metadata + labels

## Step 0: Environment Setup

> **Note:** The cell below installs `terratorch` and pins NumPy.  
> After it runs the runtime will **restart automatically**.  
> Re-run from **Step 0.2** after the restart.

In [ ]:
# Cell 0.1: Install dependencies
!pip install -q "numpy==1.26.4"
!pip install -q terratorch
!pip install -q rasterio scikit-learn geopandas matplotlib seaborn scipy pyarrow joblib

import os
print("Restarting runtime to apply numpy pin \u2014 re-run from Step 0.2 after restart.")
os.kill(os.getpid(), 9)

In [ ]:
# Cell 0.2: Clone repo
import os

REPO = 'satellite-internet-prediction-ML'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

In [ ]:
# Cell 0.3: Sync patches from Google Drive
from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

PATCH_DIR = 'data/raw/patches_2019'
Path(PATCH_DIR).mkdir(parents=True, exist_ok=True)

local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if local_count < 6900:
    print('Syncing patches from Google Drive ...')
    drive_dir = '/content/drive/MyDrive/patches_2019'
    for f in Path(drive_dir).glob('*.tif'):
        dst = Path(PATCH_DIR) / f.name
        if not dst.exists():
            shutil.copy(f, dst)
    local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
print(f'Patches available: {local_count:,}')

patch_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if patch_count < 6900:
    raise FileNotFoundError(
        f'Only {patch_count} patches found in {PATCH_DIR}. '
        f'Run NB01 first (Steps 1-7) to download patches and save them to Google Drive.'
    )

## Step 1: Imports & Constants

In [ ]:
import random, numpy as np, torch
random.seed(42); np.random.seed(42)
torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True; torch.use_deterministic_algorithms(True)

import pandas as pd
import rasterio
import torch.nn as nn
import pytorch_lightning as pl
import matplotlib.pyplot as plt
import warnings, logging, json
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score, confusion_matrix,
    classification_report, mean_absolute_error, mean_squared_error,
)
from scipy.stats import spearmanr, binned_statistic
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.getLogger('rasterio').setLevel(logging.ERROR)

HLS_MEANS = [0.14245495, 0.13921481, 0.12434631, 0.31420089, 0.20743526, 0.12046503]
HLS_STDS  = [0.04036231, 0.04186983, 0.05267646, 0.08222210, 0.06834774, 0.05294205]
SCALE     = 0.0001

PATCH_DIR   = 'data/raw/patches_2019'
RESULTS_DIR = 'outputs/results'
FIGURES_DIR = 'outputs/figures'
for d in [RESULTS_DIR, FIGURES_DIR, 'outputs/models', 'checkpoints']:
    Path(d).mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## Step 2: Load Data & Train / Val / Test Split
Pre-computed by NB01 (`patches_with_splits.csv`): binary labels, aggregate
targets, and geographic split.

In [ ]:
sampled = pd.read_csv('data/processed/patches_with_splits.csv')

available = set(f.stem for f in Path(PATCH_DIR).glob('*.tif'))
sampled = sampled[sampled['patch_id'].isin(available)].reset_index(drop=True)

train_df = sampled[sampled['split'] == 'train'].reset_index(drop=True)
val_df   = sampled[sampled['split'] == 'val'].reset_index(drop=True)
test_df  = sampled[sampled['split'] == 'test'].reset_index(drop=True)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test (NE): {len(test_df):,}')
print(f'Test binary positive rate   : {test_df["has_coverage"].mean():.2%}')
print(f'Train coverage_fraction mean: {train_df["coverage_fraction"].mean():.4f}')
print(f'Test  coverage_fraction mean: {test_df["coverage_fraction"].mean():.4f}')

## Step 3: Load Cached Prithvi Embeddings
Load frozen 1024-d Prithvi embeddings from `outputs/features/`.
If the cache is missing, extract and save them inline \u2014 this only happens once.
Subsequent epochs train only the lightweight regression head on pre-extracted embeddings.

In [ ]:
OUTPUT_FEATURES = Path('outputs/features')
OUTPUT_FEATURES.mkdir(parents=True, exist_ok=True)

DRIVE_FEATURES = Path('/content/drive/MyDrive/prithvi_embeddings')
DRIVE_FEATURES.mkdir(parents=True, exist_ok=True)


def load_or_extract_embeddings(split_name, df):
    """Load cached frozen Prithvi embeddings, or extract + cache them.
    Priority: local cache \u2192 Google Drive cache \u2192 extract fresh."""
    local_path = OUTPUT_FEATURES / f'prithvi_frozen_embeds_{split_name}.npz'
    drive_path = DRIVE_FEATURES / f'prithvi_frozen_embeds_{split_name}.npz'

    if local_path.exists():
        data = np.load(local_path)
        X = data['X'].astype(np.float32)
        print(f'  Loaded cached {split_name} (local): {X.shape}')
        return X

    if drive_path.exists():
        import shutil
        shutil.copy(drive_path, local_path)
        data = np.load(local_path)
        X = data['X'].astype(np.float32)
        print(f'  Loaded cached {split_name} (Drive): {X.shape}')
        return X

    print(f'  Cache miss for {split_name} \u2014 extracting from scratch ...')
    from terratorch.registry import BACKBONE_REGISTRY

    encoder = BACKBONE_REGISTRY.build('prithvi_eo_v2_300', pretrained=True).to(DEVICE).eval()
    for p in encoder.parameters():
        p.requires_grad = False

    class PrithviPatchDataset(Dataset):
        def __init__(self, df, patch_dir, n_bands=6):
            self.df = df.reset_index(drop=True)
            self.patch_dir = patch_dir
            self.n_bands = n_bands
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            with rasterio.open(f"{self.patch_dir}/{row['patch_id']}.tif") as src:
                img = src.read(list(range(1, self.n_bands + 1))).astype(np.float32)
            img *= SCALE
            for b in range(self.n_bands):
                img[b] = (img[b] - HLS_MEANS[b]) / HLS_STDS[b]
            img = np.nan_to_num(img, nan=0.0)
            return torch.from_numpy(img).unsqueeze(1), row['patch_id']

    ds = PrithviPatchDataset(df, PATCH_DIR)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    all_embs = []
    with torch.no_grad():
        for x, _ in tqdm(loader, desc=f'Extracting {split_name}'):
            x = x.to(DEVICE, dtype=torch.float32)
            feats = encoder(x)
            pooled = feats[-1].mean(dim=1)
            all_embs.append(pooled.cpu().numpy().astype(np.float32))

    X = np.concatenate(all_embs, axis=0)
    np.savez_compressed(local_path, X=X, patch_id=df['patch_id'].to_numpy())
    print(f'  Extracted and cached {split_name}: {X.shape}')

    import shutil
    shutil.copy(local_path, drive_path)
    print(f'  Saved to Drive: {drive_path}')

    del encoder
    torch.cuda.empty_cache()
    return X


print('Loading frozen Prithvi embeddings (1024-d) ...')
X_train_emb = load_or_extract_embeddings('train', train_df)
X_val_emb   = load_or_extract_embeddings('val',   val_df)
X_test_emb  = load_or_extract_embeddings('test',  test_df)
print(f'\nTrain: {X_train_emb.shape}  |  Val: {X_val_emb.shape}  |  Test: {X_test_emb.shape}')

In [ ]:
class EmbeddingDataset(Dataset):
    """Simple dataset wrapping pre-extracted embeddings + scalar targets."""
    def __init__(self, embeddings, targets):
        self.X = torch.from_numpy(embeddings)
        self.y = torch.tensor(targets, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

_ds = EmbeddingDataset(X_train_emb, train_df['coverage_fraction'].values)
_x, _y = _ds[0]
print(f'Embedding shape: {_x.shape}  |  Target: {_y.item():.4f}')
del _ds, _x, _y

## Step 4: Model Definitions

### Head A \u2014 Linear
`LayerNorm(1024) \u2192 Dropout(0.1) \u2192 Linear(1024, 1)`

### Head B \u2014 MLP
`LayerNorm(1024) \u2192 Linear(1024, 256) \u2192 GELU \u2192 Dropout \u2192 Linear(256, 128) \u2192 GELU \u2192 Dropout \u2192 Linear(128, 1)`

In [ ]:
class PrithviRegressor(pl.LightningModule):
    """Linear regression head on frozen Prithvi embeddings."""

    def __init__(self, target_col, lr=1e-3, weight_decay=1e-3, dropout=0.1):
        super().__init__()
        self.save_hyperparameters()
        self.head = nn.Sequential(
            nn.LayerNorm(1024),
            nn.Dropout(dropout),
            nn.Linear(1024, 1),
        )
        self.loss_fn = nn.MSELoss()
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'Target: {target_col}  |  Trainable params: {trainable:,}')

    def forward(self, x):
        return self.head(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        self.log('val_loss', self.loss_fn(self(x), y), prog_bar=True, sync_dist=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sched, 'monitor': 'val_loss'}}


class PrithviMLPRegressor(pl.LightningModule):
    """Two-hidden-layer MLP regression head on frozen Prithvi embeddings."""

    def __init__(self, target_col, lr=3e-4, weight_decay=1e-3,
                 dropout=0.2, hidden_dim_1=256, hidden_dim_2=128):
        super().__init__()
        self.save_hyperparameters()
        self.head = nn.Sequential(
            nn.LayerNorm(1024),
            nn.Linear(1024, hidden_dim_1), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim_1, hidden_dim_2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim_2, 1),
        )
        self.loss_fn = nn.MSELoss()
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f'Target: {target_col}  |  Head: 1024\u2192{hidden_dim_1}\u2192{hidden_dim_2}\u21921')
        print(f'  Trainable params: {trainable:,}')

    def forward(self, x):
        return self.head(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        self.log('val_loss', self.loss_fn(self(x), y), prog_bar=True, sync_dist=True)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=10)
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sched, 'monitor': 'val_loss'}}

## Step 5: Training & Evaluation Helpers

In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint


class MetricTracker(pl.Callback):
    def __init__(self):
        self.train_loss = []
        self.val_loss   = []
    def on_train_epoch_end(self, trainer, pl_module):
        v = trainer.callback_metrics.get('train_loss')
        self.train_loss.append(float(v) if v is not None else float('nan'))
    def on_validation_epoch_end(self, trainer, pl_module):
        v = trainer.callback_metrics.get('val_loss')
        self.val_loss.append(float(v) if v is not None else float('nan'))


def run_inference(model, embeddings, targets, batch_size=256):
    """Run head-only inference on pre-extracted embeddings."""
    dataset = EmbeddingDataset(embeddings, targets)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    preds_list = []
    model.eval()
    with torch.no_grad():
        for x, _ in loader:
            preds_list.extend(model(x.to(DEVICE)).cpu().numpy())
    return np.array(preds_list)


def plot_evaluation(test_preds, y_test_cont, y_test_bin, val_preds, y_val_bin,
                    head_name, target_name, train_loss_hist=None, val_loss_hist=None):
    """Produce 5 diagnostic figures per head \u00d7 target."""
    prefix = f'prithvi_{head_name}_agg_{target_name}'

    prec_v, rec_v, thr_v = precision_recall_curve(y_val_bin, val_preds)
    f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
    opt_thr = float(thr_v[np.argmax(f1_v)])
    y_pred_bin = (test_preds >= opt_thr).astype(int)
    residuals = test_preds - y_test_cont

    mae  = mean_absolute_error(y_test_cont, test_preds)
    rmse = np.sqrt(mean_squared_error(y_test_cont, test_preds))
    rho, _ = spearmanr(y_test_cont, test_preds)
    pr_auc = average_precision_score(y_test_bin, test_preds)
    prec_opt = precision_score(y_test_bin, y_pred_bin, zero_division=0)
    rec_opt  = recall_score(y_test_bin, y_pred_bin)

    # \u2500\u2500 Fig 1: Train + Val loss curve \u2500\u2500
    if val_loss_hist and len(val_loss_hist) > 0:
        fig, ax = plt.subplots(figsize=(8, 4))
        epochs = list(range(1, len(val_loss_hist) + 1))
        best_ep = int(np.argmin(val_loss_hist)) + 1
        ax.plot(epochs, val_loss_hist, lw=2, color='#4C72B0', label='Val loss (MSE)')
        if train_loss_hist and len(train_loss_hist) > 0:
            ax.plot(list(range(1, len(train_loss_hist) + 1)), train_loss_hist,
                    lw=2, color='#DD8452', alpha=0.7, label='Train loss (MSE)')
        ax.axvline(best_ep, ls='--', color='red', alpha=0.8,
                   label=f'Best epoch {best_ep}  (val loss = {min(val_loss_hist):.4f})')
        ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
        ax.set_title(f'Train & Validation Loss \u2014 {head_name} \u2014 {target_name}')
        ax.legend(); plt.tight_layout()
        plt.savefig(f'{FIGURES_DIR}/{prefix}_loss_curve.png', dpi=150, bbox_inches='tight')
        plt.show()

    # \u2500\u2500 Fig 2: Predicted vs actual + residual histogram \u2500\u2500
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'Regression Quality \u2014 {head_name} \u2014 {target_name}', fontsize=13, fontweight='bold')

    axes[0].scatter(y_test_cont, test_preds, alpha=0.35, s=12, color='#4C72B0')
    mn = min(y_test_cont.min(), test_preds.min())
    mx = max(y_test_cont.max(), test_preds.max())
    axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='45\u00b0 reference')
    axes[0].set_xlabel(f'Actual {target_name}'); axes[0].set_ylabel(f'Predicted {target_name}')
    axes[0].set_title(f'Predicted vs Actual\nMAE={mae:.3f}  RMSE={rmse:.3f}  Spearman \u03c1={rho:.3f}')
    axes[0].legend(fontsize=9)

    axes[1].hist(residuals, bins=40, color='#4C72B0', edgecolor='white', alpha=0.8)
    axes[1].axvline(0, color='red', ls='--', lw=1.5, label='Zero')
    axes[1].axvline(residuals.mean(), color='orange', lw=1.5, label=f'Mean = {residuals.mean():.3f}')
    axes[1].set_xlabel('Residual (pred \u2212 true)'); axes[1].set_ylabel('Count')
    axes[1].set_title(f'Residual Distribution  (std = {residuals.std():.3f})')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/{prefix}_regression.png', dpi=150, bbox_inches='tight')
    plt.show()

    # \u2500\u2500 Fig 3: Residuals vs true value \u2500\u2500
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(y_test_cont, residuals, alpha=0.35, s=12, color='#4C72B0')
    ax.axhline(0, color='red', ls='--', lw=1.5, label='Zero residual')
    try:
        bin_means, bin_edges, _ = binned_statistic(y_test_cont, residuals, statistic='mean', bins=10)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        ax.plot(bin_centers, bin_means, 'o-', color='orange', lw=2, ms=6, label='Bin mean residual')
    except Exception:
        pass
    ax.set_xlabel(f'True {target_name}'); ax.set_ylabel('Residual (pred \u2212 true)')
    ax.set_title(f'Residuals vs True Value \u2014 {head_name} \u2014 {target_name}')
    ax.legend(); plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/{prefix}_residuals.png', dpi=150, bbox_inches='tight')
    plt.show()

    # \u2500\u2500 Fig 4: Spatial prediction map \u2500\u2500
    lons, lats = test_df['lon'].values, test_df['lat'].values
    res_abs_max = np.percentile(np.abs(residuals), 95)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Spatial Prediction Map \u2014 {head_name} \u2014 {target_name}\nNE India test set',
                 fontsize=13, fontweight='bold')
    sc0 = axes[0].scatter(lons, lats, c=test_preds, cmap='RdYlGn', s=18, alpha=0.75)
    plt.colorbar(sc0, ax=axes[0], label=f'Predicted {target_name}')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude'); axes[0].set_title('Predicted value')
    sc1 = axes[1].scatter(lons, lats, c=residuals, cmap='RdBu_r', s=18, alpha=0.75,
                          vmin=-res_abs_max, vmax=res_abs_max)
    plt.colorbar(sc1, ax=axes[1], label='Residual (pred \u2212 true)')
    axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
    axes[1].set_title('Residuals (blue = under, red = over)')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/{prefix}_spatial.png', dpi=150, bbox_inches='tight')
    plt.show()

    # \u2500\u2500 Fig 5: Confusion matrix + PR curve \u2500\u2500
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Binary Connectivity Task \u2014 {head_name} \u2014 {target_name}', fontsize=13, fontweight='bold')

    cm = confusion_matrix(y_test_bin, y_pred_bin)
    im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
    fig.colorbar(im, ax=axes[0])
    axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
    axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
    for r in range(2):
        for c in range(2):
            axes[0].text(c, r, str(cm[r, c]), ha='center', va='center',
                         color='white' if cm[r, c] > cm.max() / 2 else 'black', fontsize=14)
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    axes[0].set_title(f'Confusion Matrix (thr = {opt_thr:.3f})')

    prec_t, rec_t, _ = precision_recall_curve(y_test_bin, test_preds)
    axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc:.3f}')
    axes[1].axhline(y_test_bin.mean(), ls='--', color='grey', label=f'Baseline ({y_test_bin.mean():.3f})')
    axes[1].scatter([rec_opt], [prec_opt], marker='*', s=200, color='red', zorder=5,
                    label=f'Val-opt thr = {opt_thr:.3f}')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall Curve')
    axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/{prefix}_binary.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(classification_report(y_test_bin, y_pred_bin,
                                target_names=['No Coverage', 'Has Coverage']))

## Step 6: Train & Evaluate \u2014 Linear Head

| Param | Value |
|---|---|
| `learning_rate` | 1e-3 |
| `weight_decay` | 1e-3 |
| `dropout` | 0.1 |
| `batch_size` | 32 |
| `max_epochs` | 100 |
| `early_stopping_patience` | 15 |

In [ ]:
TARGETS = ['coverage_fraction', 'log_mean_tests']

LINEAR_CFG = dict(lr=1e-3, weight_decay=1e-3, dropout=0.1)
LINEAR_TRAIN = dict(batch_size=32, max_epochs=100, patience=15)

linear_metrics = {}
linear_histories = {}

for TARGET in TARGETS:
    print(f'\n{"="*70}')
    print(f'  LINEAR \u2014 {TARGET}')
    print(f'{"="*70}')

    train_loader = DataLoader(
        EmbeddingDataset(X_train_emb, train_df[TARGET].values),
        batch_size=LINEAR_TRAIN['batch_size'], shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(
        EmbeddingDataset(X_val_emb, val_df[TARGET].values),
        batch_size=LINEAR_TRAIN['batch_size'] * 2, shuffle=False, num_workers=0, pin_memory=True)

    model = PrithviRegressor(target_col=TARGET, **LINEAR_CFG)
    tracker = MetricTracker()
    early_stop = EarlyStopping(monitor='val_loss', patience=LINEAR_TRAIN['patience'], mode='min', verbose=True)
    ckpt_cb = ModelCheckpoint(
        dirpath='checkpoints',
        filename=f'prithvi_linear_agg_{TARGET}_' + '{epoch:02d}_{val_loss:.4f}',
        monitor='val_loss', mode='min', save_top_k=1, verbose=True)

    trainer = Trainer(
        max_epochs=LINEAR_TRAIN['max_epochs'],
        accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=1,
        callbacks=[tracker, early_stop, ckpt_cb], log_every_n_steps=10, enable_progress_bar=True)
    trainer.fit(model, train_loader, val_loader)

    print(f'Best checkpoint: {ckpt_cb.best_model_path}')
    print(f'Best val loss  : {ckpt_cb.best_model_score:.6f}')
    print(f'Trained epochs : {len(tracker.val_loss)}')

    best_model = PrithviRegressor.load_from_checkpoint(ckpt_cb.best_model_path).to(DEVICE).eval()
    val_preds  = run_inference(best_model, X_val_emb, val_df[TARGET].values)
    test_preds = run_inference(best_model, X_test_emb, test_df[TARGET].values)

    # Metrics
    prec_v, rec_v, thr_v = precision_recall_curve(val_df['has_coverage'].values, val_preds)
    f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
    opt_thr = float(thr_v[np.argmax(f1_v)])
    y_pred_bin = (test_preds >= opt_thr).astype(int)
    y_test_cont = test_df[TARGET].values
    y_test_bin  = test_df['has_coverage'].values

    mae  = mean_absolute_error(y_test_cont, test_preds)
    rmse = np.sqrt(mean_squared_error(y_test_cont, test_preds))
    rho, _ = spearmanr(y_test_cont, test_preds)
    pr_auc   = average_precision_score(y_test_bin, test_preds)
    roc_auc  = roc_auc_score(y_test_bin, test_preds)
    f1_opt   = f1_score(y_test_bin, y_pred_bin)

    print(f'\n  MAE={mae:.4f}  RMSE={rmse:.4f}  Spearman={rho:.4f}')
    print(f'  PR-AUC={pr_auc:.4f}  ROC-AUC={roc_auc:.4f}  F1={f1_opt:.4f}')

    linear_metrics[TARGET] = {
        'model': f'Prithvi_300M_linear_agg_{TARGET}', 'target': TARGET,
        'mae': mae, 'rmse': rmse, 'spearman_rho': rho,
        'pr_auc': pr_auc, 'roc_auc': roc_auc, 'f1_opt': f1_opt,
        'opt_threshold': opt_thr,
        'precision_opt': precision_score(y_test_bin, y_pred_bin, zero_division=0),
        'recall_opt': recall_score(y_test_bin, y_pred_bin),
        'accuracy_opt': accuracy_score(y_test_bin, y_pred_bin),
        'n_train': len(train_df) + len(val_df), 'n_test': len(test_df),
    }
    linear_histories[TARGET] = {'train': tracker.train_loss, 'val': tracker.val_loss}

    plot_evaluation(test_preds, y_test_cont, y_test_bin, val_preds, val_df['has_coverage'].values,
                    head_name='linear', target_name=TARGET,
                    train_loss_hist=tracker.train_loss, val_loss_hist=tracker.val_loss)

    del best_model, trainer, model
    torch.cuda.empty_cache()

print('\nLinear head training complete.')

## Step 7: Train & Evaluate \u2014 MLP Head

| Param | Value |
|---|---|
| `hidden_dim_1` | 256 |
| `hidden_dim_2` | 128 |
| `learning_rate` | 3e-4 |
| `weight_decay` | 1e-3 |
| `dropout` | 0.2 |
| `activation` | GELU |
| `batch_size` | 32 |
| `max_epochs` | 150 |
| `early_stopping_patience` | 15 |

In [ ]:
MLP_CFG = dict(lr=3e-4, weight_decay=1e-3, dropout=0.2, hidden_dim_1=256, hidden_dim_2=128)
MLP_TRAIN = dict(batch_size=32, max_epochs=150, patience=15)

mlp_metrics = {}
mlp_histories = {}

for TARGET in TARGETS:
    print(f'\n{"="*70}')
    print(f'  MLP \u2014 {TARGET}')
    print(f'{"="*70}')

    train_loader = DataLoader(
        EmbeddingDataset(X_train_emb, train_df[TARGET].values),
        batch_size=MLP_TRAIN['batch_size'], shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(
        EmbeddingDataset(X_val_emb, val_df[TARGET].values),
        batch_size=MLP_TRAIN['batch_size'] * 2, shuffle=False, num_workers=0, pin_memory=True)

    model = PrithviMLPRegressor(target_col=TARGET, **MLP_CFG)
    tracker = MetricTracker()
    early_stop = EarlyStopping(monitor='val_loss', patience=MLP_TRAIN['patience'], mode='min', verbose=True)
    ckpt_cb = ModelCheckpoint(
        dirpath='checkpoints',
        filename=f'prithvi_mlp_agg_{TARGET}_' + '{epoch:02d}_{val_loss:.4f}',
        monitor='val_loss', mode='min', save_top_k=1, verbose=True)

    trainer = Trainer(
        max_epochs=MLP_TRAIN['max_epochs'],
        accelerator='gpu' if torch.cuda.is_available() else 'cpu', devices=1,
        callbacks=[tracker, early_stop, ckpt_cb], log_every_n_steps=10, enable_progress_bar=True)
    trainer.fit(model, train_loader, val_loader)

    print(f'Best checkpoint: {ckpt_cb.best_model_path}')
    print(f'Best val loss  : {ckpt_cb.best_model_score:.6f}')
    print(f'Trained epochs : {len(tracker.val_loss)}')

    best_model = PrithviMLPRegressor.load_from_checkpoint(ckpt_cb.best_model_path).to(DEVICE).eval()
    val_preds  = run_inference(best_model, X_val_emb, val_df[TARGET].values)
    test_preds = run_inference(best_model, X_test_emb, test_df[TARGET].values)

    prec_v, rec_v, thr_v = precision_recall_curve(val_df['has_coverage'].values, val_preds)
    f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
    opt_thr = float(thr_v[np.argmax(f1_v)])
    y_pred_bin = (test_preds >= opt_thr).astype(int)
    y_test_cont = test_df[TARGET].values
    y_test_bin  = test_df['has_coverage'].values

    mae  = mean_absolute_error(y_test_cont, test_preds)
    rmse = np.sqrt(mean_squared_error(y_test_cont, test_preds))
    rho, _ = spearmanr(y_test_cont, test_preds)
    pr_auc   = average_precision_score(y_test_bin, test_preds)
    roc_auc  = roc_auc_score(y_test_bin, test_preds)
    f1_opt   = f1_score(y_test_bin, y_pred_bin)

    print(f'\n  MAE={mae:.4f}  RMSE={rmse:.4f}  Spearman={rho:.4f}')
    print(f'  PR-AUC={pr_auc:.4f}  ROC-AUC={roc_auc:.4f}  F1={f1_opt:.4f}')

    mlp_metrics[TARGET] = {
        'model': f'Prithvi_300M_mlp_agg_{TARGET}', 'target': TARGET,
        'mae': mae, 'rmse': rmse, 'spearman_rho': rho,
        'pr_auc': pr_auc, 'roc_auc': roc_auc, 'f1_opt': f1_opt,
        'opt_threshold': opt_thr,
        'precision_opt': precision_score(y_test_bin, y_pred_bin, zero_division=0),
        'recall_opt': recall_score(y_test_bin, y_pred_bin),
        'accuracy_opt': accuracy_score(y_test_bin, y_pred_bin),
        'n_train': len(train_df) + len(val_df), 'n_test': len(test_df),
    }
    mlp_histories[TARGET] = {'train': tracker.train_loss, 'val': tracker.val_loss}

    plot_evaluation(test_preds, y_test_cont, y_test_bin, val_preds, val_df['has_coverage'].values,
                    head_name='mlp', target_name=TARGET,
                    train_loss_hist=tracker.train_loss, val_loss_hist=tracker.val_loss)

    del best_model, trainer, model
    torch.cuda.empty_cache()

print('\nMLP head training complete.')

## Step 8: Summary \u2014 Linear vs MLP Comparison

In [ ]:
rows = []
for TARGET in TARGETS:
    for head, m_dict in [('Linear', linear_metrics), ('MLP', mlp_metrics)]:
        m = m_dict[TARGET]
        rows.append({
            'Head': head, 'Target': TARGET,
            'MAE': m['mae'], 'RMSE': m['rmse'], 'Spearman \u03c1': m['spearman_rho'],
            'PR-AUC': m['pr_auc'], 'ROC-AUC': m['roc_auc'], 'F1': m['f1_opt'],
        })

summary_df = pd.DataFrame(rows)
print('=== Head-Capacity Ablation: Linear vs MLP ===')
print(summary_df.to_string(index=False))

In [ ]:
# \u2500\u2500 RMSE comparison bar chart \u2500\u2500
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Prithvi-300M Head-Capacity Ablation \u2014 NE India Test Set',
             fontsize=14, fontweight='bold')

heads = ['Linear', 'MLP']
colors = {'Linear': '#4C72B0', 'MLP': '#DD8452'}

# Left panel: regression metrics (MAE, RMSE, Spearman)
ax = axes[0]
x = np.arange(len(TARGETS))
w = 0.35
for i, head in enumerate(heads):
    rmse_vals = [linear_metrics[t]['rmse'] if head == 'Linear' else mlp_metrics[t]['rmse'] for t in TARGETS]
    mae_vals  = [linear_metrics[t]['mae']  if head == 'Linear' else mlp_metrics[t]['mae']  for t in TARGETS]
    bars = ax.bar(x + i * w, rmse_vals, w, label=f'{head} RMSE', color=colors[head], alpha=0.85)
    for j, (rv, mv) in enumerate(zip(rmse_vals, mae_vals)):
        ax.text(x[j] + i * w, rv + 0.002, f'{rv:.4f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x + w / 2)
ax.set_xticklabels(TARGETS, fontsize=10)
ax.set_ylabel('RMSE (lower is better)')
ax.set_title('Regression: RMSE')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

# Right panel: binary task metrics
ax = axes[1]
metrics_names = ['PR-AUC', 'ROC-AUC', 'F1']
x2 = np.arange(len(TARGETS))
bar_w = 0.15
for i, head in enumerate(heads):
    m_d = linear_metrics if head == 'Linear' else mlp_metrics
    for j, mn in enumerate(metrics_names):
        key = mn.lower().replace('-', '_').replace(' ', '_')
        if key == 'f1': key = 'f1_opt'
        vals = [m_d[t][key] for t in TARGETS]
        offset = (i * len(metrics_names) + j) * bar_w
        label = f'{head} {mn}' if i == 0 or j == 0 else None
        c = colors[head]
        alpha = 1.0 - j * 0.2
        ax.bar(x2 + offset, vals, bar_w, color=c, alpha=alpha,
               label=f'{head} {mn}')

ax.set_xticks(x2 + bar_w * 2.5)
ax.set_xticklabels(TARGETS, fontsize=10)
ax.set_ylabel('Score (higher is better)')
ax.set_title('Binary Task Metrics')
ax.set_ylim(0, 1)
ax.legend(fontsize=7, ncol=2)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/prithvi_head_ablation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8b: Save Outputs to Google Drive
Persist all generated outputs (figures, metrics) to Google Drive so they
survive Colab runtime disconnects.

In [ ]:
DRIVE_OUTPUT = '/content/drive/MyDrive/satellite-internet-outputs/NB04'
Path(DRIVE_OUTPUT).mkdir(parents=True, exist_ok=True)

for folder in ['outputs/results', 'outputs/figures']:
    src = Path(folder)
    if src.exists():
        dst = Path(DRIVE_OUTPUT) / folder
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'  Copied {folder} → {dst}')

print(f'\nAll NB04 outputs saved to Google Drive: {DRIVE_OUTPUT}')

## Step 9: Save Metrics & Push to GitHub (author only)
This cell is used by the author to commit results back to the repository.
Reviewers/reproducers can skip this step.

In [ ]:
for TARGET in TARGETS:
    pd.DataFrame([linear_metrics[TARGET]]).to_csv(
        f'{RESULTS_DIR}/prithvi_linear_agg_{TARGET}_metrics.csv', index=False)
    pd.DataFrame([mlp_metrics[TARGET]]).to_csv(
        f'{RESULTS_DIR}/prithvi_mlp_agg_{TARGET}_metrics.csv', index=False)

summary_df.to_csv(f'{RESULTS_DIR}/prithvi_head_ablation_summary.csv', index=False)
print('All metrics saved.')

In [ ]:
import subprocess, os

token = "YOUR_TOKEN_HERE"
repo  = "tatyana21111/satellite-internet-prediction-ML"

subprocess.run(['git', 'config', '--global', 'user.email', 'tatyana211zy@outlook.com'], check=True)
subprocess.run(['git', 'config', '--global', 'user.name',  'tatyana21111'], check=True)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/{repo}.git'], check=True)

subprocess.run(['git', 'add', '-f', 'notebooks/04_prithvi_head_ablation.ipynb'], check=True)

for pat in ['prithvi_linear_agg_*.csv', 'prithvi_mlp_agg_*.csv', 'prithvi_head_ablation_summary.csv']:
    for p in Path(RESULTS_DIR).glob(pat):
        subprocess.run(['git', 'add', '-f', str(p)], check=True)
        print(f'  Staged: {p}')

for pat in ['prithvi_linear_agg_*.png', 'prithvi_mlp_agg_*.png', 'prithvi_head_ablation_*.png']:
    for p in Path(FIGURES_DIR).glob(pat):
        subprocess.run(['git', 'add', '-f', str(p)], check=True)
        print(f'  Staged: {p}')

diff = subprocess.run(['git', 'diff', '--cached', '--quiet'], capture_output=True)
if diff.returncode != 0:
    subprocess.run(['git', 'commit', '-m',
                    'NB04: Prithvi-300M head-capacity ablation \u2014 Linear + MLP'], check=True)
else:
    print('Nothing staged.')

subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=True)
subprocess.run(['git', 'push'], check=True)
print('Push successful.')